# 55 Input & Result (100% datos) — Estimador de Precio

Igual que `55_input_result` pero entrenando sobre **todas las observaciones**.

| Modelo | Fuente | CV-RMSE | Test R² |
|---|---|---|---|
| XGBoost Sale (`53_boost_sale_optuna`) | `params_sale.json` | 0.233 | ≈ 0.85 |
| XGBoost Rent (`53_boost_rent`) | `params_rent.json` | 0.146 | ≈ 0.62 |

**Intervalo de error:** se usa el **CV-RMSE** (5-fold) del JSON, más robusto que el RMSE de un split 20%.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "gold").exists():
            return p
    raise FileNotFoundError("No se encontró la raíz del proyecto (data/gold)")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PARAMS_DIR   = PROJECT_ROOT / "data" / "model_results"
print(f"Proyecto: {PROJECT_ROOT}")


Proyecto: /Users/sitomachucas/Documents/BezanillaSL


In [2]:
sale_cfg = json.loads((PARAMS_DIR / "params_sale.json").read_text(encoding="utf-8"))
rent_cfg = json.loads((PARAMS_DIR / "params_rent.json").read_text(encoding="utf-8"))

RANDOM_STATE = sale_cfg["random_state"]
TEST_SIZE    = sale_cfg["test_size"]

SALE_TARGET_COL     = sale_cfg["target_col"]
SALE_IQR_FACTOR     = sale_cfg["iqr_factor"]
SALE_MIN_MUNI_OBS   = sale_cfg["min_muni_obs"]
SALE_PRECIO_M2_MIN  = sale_cfg["precio_m2_min"]
SALE_BASE_FEATURES  = sale_cfg["base_features"]
XGB_PARAMS_SALE     = {**sale_cfg["xgb_params"],
                       "random_state": RANDOM_STATE, "n_jobs": -1, "verbosity": 0}

RENT_TARGET_COL           = rent_cfg["target_col"]
RENT_IQR_FACTOR           = rent_cfg["iqr_factor"]
RENT_MIN_MUNI_OBS         = rent_cfg["min_muni_obs"]
RENT_PRECIO_M2_VACACIONAL = rent_cfg["precio_m2_vacacional_umbral"]
RENT_PRECIO_M2_MIN        = rent_cfg["precio_m2_min_umbral"]
RENT_BASE_FEATURES        = rent_cfg["base_features"]
XGB_PARAMS_RENT           = {**rent_cfg["xgb_params"],
                              "random_state": RANDOM_STATE, "n_jobs": -1, "verbosity": 0}

SALE_PATH = PROJECT_ROOT / "data" / "gold" / "final_sale_idealistaAPI.csv"
RENT_PATH = PROJECT_ROOT / "data" / "gold" / "final_rent_idealistaAPI.csv"

s_nb  = sale_cfg["notebook"];  r_nb  = rent_cfg["notebook"]
s_tri = sale_cfg["optuna_best_trial"]; r_tri = rent_cfg["optuna_best_trial"]
s_cv  = sale_cfg["optuna_cv_rmse"];   r_cv  = rent_cfg["optuna_cv_rmse"]
s_r2  = sale_cfg["test_r2"];          r_r2  = rent_cfg["test_r2"]
print(f"Sale [{s_nb}] trial #{s_tri}  CV-RMSE={s_cv}  Test R\u00b2={s_r2}")
print(f"Rent [{r_nb}] trial #{r_tri}  CV-RMSE={r_cv}  Test R\u00b2={r_r2}")


Sale [53_boost_sale_optuna] trial #76  CV-RMSE=0.23347  Test R²=0.8531
Rent [53_boost_rent] trial #97  CV-RMSE=0.14622  Test R²=0.6179


In [3]:
def build_X(df: pd.DataFrame, base_features: list, min_muni_obs: int) -> tuple:
    df2 = df.copy()
    base = [f for f in base_features if f in df2.columns]
    mun_cols = sorted([c for c in df2.columns if c.startswith("municipio_")])
    if mun_cols:
        counts = df2[mun_cols].sum()
        small  = counts[counts < min_muni_obs].index.tolist()
        if small:
            df2["municipio_otros"] = df2[small].max(axis=1)
            df2 = df2.drop(columns=small)
        mun_final = sorted(c for c in df2.columns if c.startswith("municipio_"))
    else:
        mun_final = []
    all_feats = base + [m for m in mun_final if m not in base]
    X_raw = df2[all_feats].copy()
    medians = X_raw.median().to_dict()
    imputer = SimpleImputer(strategy="median")
    X = pd.DataFrame(imputer.fit_transform(X_raw), columns=X_raw.columns, index=X_raw.index)
    return X, all_feats, medians

print("Funciones cargadas.")


Funciones cargadas.


In [4]:
SALE_CV_RMSE = sale_cfg["optuna_cv_rmse"]
RENT_CV_RMSE = rent_cfg["optuna_cv_rmse"]

# ── SALE ─────────────────────────────────────────────────────────────────────
df_sale = pd.read_csv(SALE_PATH)
df_sale = df_sale[df_sale[SALE_TARGET_COL].notna()].copy()
q1, q3 = df_sale[SALE_TARGET_COL].quantile([0.25, 0.75])
iqr = q3 - q1
df_sale = df_sale[df_sale[SALE_TARGET_COL].between(q1 - SALE_IQR_FACTOR*iqr, q3 + SALE_IQR_FACTOR*iqr)].copy()
df_sale = df_sale[df_sale["precio_m2"] >= SALE_PRECIO_M2_MIN].copy()
print(f"Sale  \u2014 filas tras limpieza: {len(df_sale)}")

X_sale, feats_sale, medians_sale = build_X(df_sale, SALE_BASE_FEATURES, SALE_MIN_MUNI_OBS)
y_sale = df_sale[SALE_TARGET_COL].values
model_sale = XGBRegressor(**XGB_PARAMS_SALE)
model_sale.fit(X_sale, y_sale)
print(f"Sale  \u2014 100% ({len(feats_sale)} features)  |  CV-RMSE: {SALE_CV_RMSE}  \u2192  \u00b1{(np.exp(SALE_CV_RMSE)-1)*100:.1f}%")

# ── RENT ─────────────────────────────────────────────────────────────────────
df_rent = pd.read_csv(RENT_PATH)
df_rent = df_rent[df_rent[RENT_TARGET_COL].notna() & df_rent["precio_m2"].notna()].copy()
df_rent = df_rent[df_rent["precio_m2"] <= RENT_PRECIO_M2_VACACIONAL].copy()
df_rent = df_rent[df_rent["precio_m2"] >= RENT_PRECIO_M2_MIN].copy()
q1, q3 = df_rent[RENT_TARGET_COL].quantile([0.25, 0.75])
iqr = q3 - q1
df_rent = df_rent[df_rent[RENT_TARGET_COL].between(q1 - RENT_IQR_FACTOR*iqr, q3 + RENT_IQR_FACTOR*iqr)].copy()
print(f"Rent  \u2014 filas tras limpieza: {len(df_rent)}")

X_rent, feats_rent, medians_rent = build_X(df_rent, RENT_BASE_FEATURES, RENT_MIN_MUNI_OBS)
y_rent = df_rent[RENT_TARGET_COL].values
model_rent = XGBRegressor(**XGB_PARAMS_RENT)
model_rent.fit(X_rent, y_rent)
print(f"Rent  \u2014 100% ({len(feats_rent)} features)  |  CV-RMSE: {RENT_CV_RMSE}  \u2192  \u00b1{(np.exp(RENT_CV_RMSE)-1)*100:.1f}%")
print("\n\u2713 Ambos modelos listos.")


Sale  — filas tras limpieza: 2543
Sale  — 100% (52 features)  |  CV-RMSE: 0.23347  →  ±26.3%
Rent  — filas tras limpieza: 662
Rent  — 100% (27 features)  |  CV-RMSE: 0.14622  →  ±15.7%

✓ Ambos modelos listos.


In [5]:
GEO_COLS = [
    "distancia_min_playa_km", "distancia_min_supermercado_km",
    "distancia_min_colegio_km", "precio_m2_municipio_media",
    "distancia_centro_municipio_km", "score_cercania_servicios",
    "latitud", "longitud",
]

def build_geo_ref(df: pd.DataFrame) -> pd.DataFrame:
    mun_cols = [c for c in df.columns if c.startswith("municipio_")]
    rows = []
    for mc in mun_cols:
        nombre = mc.replace("municipio_", "")
        subset = df[df[mc] == 1]
        if len(subset) == 0:
            continue
        row = {"municipio": nombre}
        for gc in GEO_COLS:
            row[gc] = subset[gc].median() if gc in subset.columns else np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index("municipio")

sale_geo_ref = build_geo_ref(df_sale)
rent_geo_ref = build_geo_ref(df_rent)
print(f"Referencia geográfica \u2014 Sale: {len(sale_geo_ref)} municipios")
print(f"Referencia geográfica \u2014 Rent: {len(rent_geo_ref)} municipios")


Referencia geográfica — Sale: 31 municipios
Referencia geográfica — Rent: 10 municipios


In [6]:
print("Municipios disponibles (VENTA):")
print(sorted(sale_geo_ref.index.tolist()))
print()
print("Municipios disponibles (ALQUILER):")
print(sorted(rent_geo_ref.index.tolist()))


Municipios disponibles (VENTA):
['Ampuero', 'Barcena de Cicero', 'Camargo', 'Castro-Urdiales', 'Colindres', 'Cudon', 'El Astillero', 'Guarnizo', 'Laredo', 'Liendo', 'Limpias', 'Marina de Cudeyo', 'Miengo', 'Mogro', 'Noja', 'Ortuella', 'Piélagos', 'Polanco', 'Ribamontan al Mar', 'Ribamontan al Monte', 'Santa Cruz de Bezana', 'Santander', 'Santillana del Mar', 'Santoña', 'Santurtzi', 'Suances', 'Torrelavega', 'Villaescusa', 'Viveda', 'Voto', 'otro']

Municipios disponibles (ALQUILER):
['Camargo', 'Castro-Urdiales', 'El Astillero', 'Laredo', 'Piélagos', 'Santa Cruz de Bezana', 'Santander', 'Suances', 'Torrelavega', 'otro']


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
#  ESTIMADOR DE PRECIO — modifica los valores y ejecuta esta celda
# ══════════════════════════════════════════════════════════════════════════════
MUNICIPIO      = "Santa Cruz de Bezana"   # Nombre exacto (ver celda anterior)
SUPERFICIE_M2  = 240            # m² construidos
N_DORMITORIOS  = 5             # Número de dormitorios
N_BANOS        = 3             # Número de baños
TIENE_GARAJE   = True          # True / False
OBRA_NUEVA     = True         # True / False
TIPOLOGIA      = "unifamiliar"        # "piso"  o  "unifamiliar"

# ── Solo para PISO (pon None si es unifamiliar) ───────────────────────────────
PLANTA         = None             # 0=bajo, 1=primera…  → None si unifamiliar
ES_EXTERIOR    = None          # True / False         → None si unifamiliar
TIENE_ASCENSOR = None          # True / False         → None si unifamiliar


def _build_row(municipio, sup, dorm, banos, planta, exterior, ascensor,
               garaje, obra_nueva, tipologia, feature_cols, geo_ref, medians):
    row = pd.Series(np.nan, index=feature_cols)

    def _set(k, v):
        if k in row.index and v is not None:
            row[k] = float(v)

    _set("superficie_construida_m2",       sup)
    _set("numero_dormitorios",             dorm)
    _set("numero_banos",                   banos)
    _set("tiene_garaje",                   int(garaje))
    _set("obra_nueva",                     int(obra_nueva))
    _set("tipologia_unificada_piso",       1 if tipologia == "piso"        else 0)
    _set("tipologia_unificada_unifamiliar", 1 if tipologia == "unifamiliar" else 0)
    _set("planta_num",                     planta)
    _set("es_exterior_piso",    int(exterior) if exterior is not None else None)
    _set("tiene_ascensor_piso", int(ascensor) if ascensor is not None else None)

    if tipologia == "piso" and planta is not None and ascensor is not None:
        _set("interaccion_planta_sin_ascensor_piso", planta * (1 - int(ascensor)))
    else:
        _set("interaccion_planta_sin_ascensor_piso", 0)

    # ratio_dormitorios_superficie y ratio_banos_superficie se dejan como NaN
    # → relleno con mediana de entrenamiento abajo.
    # Calcularlos desde el input (dorm/sup) invierte la relación cuando sube
    # la superficie: ratio baja, el modelo predice MENOS alquiler → no monótono.

    if municipio in geo_ref.index:
        for col, val in geo_ref.loc[municipio].to_dict().items():
            _set(col, val)

    for c in [c for c in feature_cols if c.startswith("municipio_")]:
        row[c] = 0.0
    mun_col = "municipio_" + municipio
    if mun_col in row.index:
        row[mun_col] = 1.0
    elif "municipio_otros" in row.index:
        row["municipio_otros"] = 1.0

    for col in feature_cols:
        if pd.isna(row[col]):
            row[col] = medians.get(col, 0)
    return pd.DataFrame([row])


def _rango(precio, rmse_log):
    return precio * np.exp(-rmse_log), precio * np.exp(rmse_log)


def estimar_precio(municipio, sup, dorm, banos, planta, exterior,
                   ascensor, garaje, obra_nueva, tipologia):
    desc = []
    if tipologia == "piso":
        if planta   is not None: desc.append(f"Planta {planta}")
        if exterior is not None: desc.append("exterior" if exterior else "interior")
        if ascensor is not None: desc.append("con ascensor" if ascensor else "sin ascensor")
    if garaje:     desc.append("garaje")
    if obra_nueva: desc.append("obra nueva")

    print()
    print("\u2550" * 58)
    print(f"  {sup} m\u00b2  \u00b7  {dorm} dorm.  \u00b7  {banos} ba\u00f1os  \u2014  {municipio}")
    print(f"  {tipologia.upper()}  \u00b7  {' \u00b7 '.join(desc) if desc else '\u2014'}")
    print("\u2550" * 58)

    if municipio not in sale_geo_ref.index:
        print(f"\n  \u26a0  '{municipio}' no disponible en venta")
    else:
        X_pred = _build_row(municipio, sup, dorm, banos, planta, exterior,
                            ascensor, garaje, obra_nueva, tipologia,
                            feats_sale, sale_geo_ref, medians_sale)
        pv = np.exp(model_sale.predict(X_pred)[0])
        lo, hi = _rango(pv, SALE_CV_RMSE)
        pct = (np.exp(SALE_CV_RMSE) - 1) * 100
        print(f"\n  Precio de venta estimado   : {pv:>12,.0f} \u20ac")
        print(f"  Intervalo error (\u00b11\u03c3)      : [{lo:>10,.0f} \u20ac  \u2014  {hi:>10,.0f} \u20ac]  (\u00b1{pct:.0f}%)")

    if municipio not in rent_geo_ref.index:
        print(f"  \u26a0  '{municipio}' no disponible en alquiler")
    else:
        X_pred = _build_row(municipio, sup, dorm, banos, planta, exterior,
                            ascensor, garaje, obra_nueva, tipologia,
                            feats_rent, rent_geo_ref, medians_rent)
        pr = np.exp(model_rent.predict(X_pred)[0])
        lo, hi = _rango(pr, RENT_CV_RMSE)
        pct = (np.exp(RENT_CV_RMSE) - 1) * 100
        print(f"\n  Alquiler mensual estimado  : {pr:>12,.0f} \u20ac/mes")
        print(f"  Intervalo error (\u00b11\u03c3)      : [{lo:>10,.0f} \u20ac/mes  \u2014  {hi:>10,.0f} \u20ac/mes]  (\u00b1{pct:.0f}%)")

    print()


estimar_precio(
    MUNICIPIO, SUPERFICIE_M2, N_DORMITORIOS, N_BANOS,
    PLANTA, ES_EXTERIOR, TIENE_ASCENSOR, TIENE_GARAJE, OBRA_NUEVA, TIPOLOGIA
)



══════════════════════════════════════════════════════════
  240 m²  ·  5 dorm.  ·  3 baños  —  Santa Cruz de Bezana
  UNIFAMILIAR  ·  garaje · obra nueva
══════════════════════════════════════════════════════════

  Precio de venta estimado   :      678,417 €
  Intervalo error (±1σ)      : [   537,158 €  —     856,823 €]  (±26%)

  Alquiler mensual estimado  :        1,565 €/mes
  Intervalo error (±1σ)      : [     1,352 €/mes  —       1,811 €/mes]  (±16%)

